# env_19 (Command15) — notebook completo, protocolo reproducible para RA-L

Orden de ejecución (cada bloque `# %%` es una celda):

| # | Celda | Notas |
|---|-------|-------|
| 0 | Configuración | única celda con rutas/constantes |
| 1 | Generar cfgs | reproducible (`GRID_SEED`) + `grid_manifest.json` |
| 2 | Lanzador + estado | `run_cmd` con resumen en vivo, `train_one` validado, `save_state/load_state` |
| 3–4 | Smoke test | entrenamiento (3 it.) y `play.py`, en `_smoke/` (no toca el grid) |
| 5–6 | Etapa 1 y shortlist | 35 configs × 1 semilla; shortlist solo con train |
| 7–9 | Etapa 2, ablación, no-DR | multi-semilla, 1500 it. fijas |
| 10 | (tras reiniciar kernel) | recargar estado |
| 11 | Evaluación | matriz de condiciones, validada |
| 12–16 | Métricas, tablas, figuras | IC bootstrap sobre semillas de entrenamiento |
| 17 | Manifiesto | material suplementario |

Requisitos en disco: `dynamics_utils.py` junto a `env.py` (env_19); `eval_utils.py` junto a `play.py`;
`play.py`, `MyOnPolicyRunner.py` y `train.py` con los parches ya aplicados.

## 0 · Configuración

In [ ]:
import os, sys, json, time, random, re, shutil, subprocess
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path("C:/Users/Teresa/Documents/GitHub/navsim")
experiment_name = "command15"          # -> ppo_random_command15_{i}
task_name       = "Command15"          # -> Isaac-Command15-Aerotaxi-RANDOM-{i}  (registrado en __init__.py -> env_19)
cfg_dir         = ROOT / "isaac_lab/aerotaxi/env_command/env_19/grid_hiperparametros/cfg_files"
path_resultados = ROOT / "tmp/tmpTeresa/resultados_grid_command_v19"
path_ral        = ROOT / "tmp/tmpTeresa/resultados_ral_v19"
isaaclab_bat    = "C:/Users/Teresa/Desktop/RuralData/IsaacLab/IsaacLab/isaaclab.bat"
train_script    = str(ROOT / "isaac_lab/rl_v5_1_0/rsl_rl/train.py")
play_script     = str(ROOT / "isaac_lab/rl_v5_1_0/rsl_rl/play.py")
id_task         = f"Isaac-{task_name}-Aerotaxi-RANDOM"

NUM_ENVS_TRAIN    = 8192               # antes 8196 (typo de 2**13)
NUM_STEPS_PER_ENV = 128                # única fuente de verdad: el RunnerCfg
NUM_ITER          = 4000               # etapa 1 (int, no str)
NUM_ITER_STAGE2   = 1500               # presupuesto FIJO e igual para todas las semillas/brazos
N_CONFIGS         = 35
GRID_SEED         = 19                 # muestreo de hiperparámetros reproducible
REUSE_ENV17_CFGS  = False              # True: mismos hiperparámetros que env_17 (comparación antes/después)
ENV17_CFG_DIR     = ROOT / "isaac_lab/aerotaxi/env_command/env_17/grid_hiperparametros/cfg_files"

TRAIN_SEEDS  = [11, 22, 33, 44, 55]    # unidad estadística = semilla de entrenamiento (>=5)
TOP_K        = 3                       # configs que pasan a etapa 2
EVAL_SEED    = 1001                    # distinta de las de entrenamiento
N_TEST_ENVS  = 1024
DT           = 0.02                    # env.step_dt
BURN_IN_S    = 5.0                     # transitorio excluido de RMSE_ss

# Índices gym (el __init__ registra 1..100):
#   1..35 etapa 1 · 36..(35+5*TOP_K) etapa 2 · siguientes 10 ablación · siguientes 5 no-DR · 100 smoke
SMOKE_INDEX = 100

for d in (cfg_dir, path_resultados, path_ral, path_resultados / "logs", path_ral / "logs"):
    d.mkdir(parents=True, exist_ok=True)

## 1 · Generar configuraciones (reproducible + manifiesto)

In [ ]:
param_options = {
    "init_noise_std":     [0.1, 0.2, 0.5, 0.8, 1.0],
    "actor_hidden_dims":  [[64, 64], [128, 128], [256, 256], [128, 64]],
    "critic_hidden_dims": [[64, 64], [128, 128], [256, 256]],
    "activation":         ["relu", "tanh", "elu", "gelu"],
    "value_loss_coef":    [0.5, 1.0, 2.0],
    "clip_param":         [0.1, 0.2, 0.3],
    "entropy_coef":       [0.0, 0.001, 0.01, 0.05],
    "num_learning_epochs":[4, 8, 12],
    "num_mini_batches":   [2, 4, 8],
    "learning_rate":      [5e-05, 1e-4, 3e-4, 1e-3, 3e-3],
    "gamma":              [0.95, 0.98, 0.99],
    "lam":                [0.90, 0.95, 0.97],
    "max_grad_norm":      [0.5, 1.0, 2.0],
    "desired_kl":         [0.005, 0.008, 0.01, 0.02],
}

CFG_TEMPLATE = '''from isaaclab.utils import configclass
from isaaclab_rl.rsl_rl import RslRlOnPolicyRunnerCfg, RslRlPpoActorCriticCfg, RslRlPpoAlgorithmCfg

@configclass
class {task_name}PPORunnerCfg(RslRlOnPolicyRunnerCfg):
    seed = {seed}
    num_steps_per_env = {num_steps}
    max_iterations = {num_iter}
    save_interval = 100
    experiment_name = "{experiment_name}"
    run_name = "{run_name}"
    resume = False
    empirical_normalization = True
    csv_path_metrics = "{csv_path}"
    callbacks = True
    early_stopping_patience = {patience}      # 0 = desactivado (requiere el runner parcheado)
    policy = RslRlPpoActorCriticCfg(
        init_noise_std={init_noise_std},
        actor_hidden_dims={actor_hidden_dims},
        critic_hidden_dims={critic_hidden_dims},
        activation="{activation}",
    )
    algorithm = RslRlPpoAlgorithmCfg(
        value_loss_coef={value_loss_coef},
        use_clipped_value_loss=True,
        clip_param={clip_param},
        entropy_coef={entropy_coef},
        num_learning_epochs={num_learning_epochs},
        num_mini_batches={num_mini_batches},
        learning_rate={learning_rate},
        schedule="adaptive",
        gamma={gamma},
        lam={lam},
        desired_kl={desired_kl},
        max_grad_norm={max_grad_norm},
    )
'''

def write_cfg(index: int, hp: dict, seed: int, csv_path: Path = path_resultados, num_iter: int | None = None,
              patience: int = 500) -> str:
    run_name = f"ppo_random_{experiment_name}_{index}"
    body = CFG_TEMPLATE.format(task_name=task_name, seed=seed, num_steps=NUM_STEPS_PER_ENV, num_iter=num_iter or NUM_ITER,
                               experiment_name=experiment_name, run_name=run_name,
                               csv_path=str(csv_path).replace("\\", "/"), patience=patience, **hp)
    (cfg_dir / f"{run_name}_cfg.py").write_text(body, encoding="utf-8")
    return run_name

manifest_path = cfg_dir / "grid_manifest.json"
if manifest_path.exists():
    print("grid_manifest.json ya existe -> NO se regenera (evita sobrescribir el diseño del experimento).")
    grid = json.loads(manifest_path.read_text())
else:
    rng = random.Random(GRID_SEED)
    grid = {}
    for i in range(1, N_CONFIGS + 1):
        hp = {k: rng.choice(v) for k, v in param_options.items()}
        if REUSE_ENV17_CFGS:
            old = (ENV17_CFG_DIR / f"ppo_random_command13_{i}_cfg.py").read_text(encoding="utf-8")
            pick = lambda pat, cast=float: cast(re.search(pat, old).group(1))
            hp = dict(
                init_noise_std=pick(r"init_noise_std=([\d.e-]+)"),
                actor_hidden_dims=json.loads(re.search(r"actor_hidden_dims=(\[.*?\])", old).group(1)),
                critic_hidden_dims=json.loads(re.search(r"critic_hidden_dims=(\[.*?\])", old).group(1)),
                activation=re.search(r'activation="(\w+)"', old).group(1),
                value_loss_coef=pick(r"value_loss_coef=([\d.e-]+)"), clip_param=pick(r"clip_param=([\d.e-]+)"),
                entropy_coef=pick(r"entropy_coef=([\d.e-]+)"), num_learning_epochs=pick(r"num_learning_epochs=(\d+)", int),
                num_mini_batches=pick(r"num_mini_batches=(\d+)", int), learning_rate=pick(r"learning_rate=([\d.e-]+)"),
                gamma=pick(r"gamma=([\d.e-]+)"), lam=pick(r"lam=([\d.e-]+)"),
                desired_kl=pick(r"desired_kl=([\d.e-]+)"), max_grad_norm=pick(r"max_grad_norm=([\d.e-]+)"))
        grid[str(i)] = {"hp": hp, "seed": 42}                  # etapa 1: UNA semilla común
        write_cfg(i, hp, seed=42)
    manifest_path.write_text(json.dumps(grid, indent=1))
print(f"{len(grid)} configuraciones en {cfg_dir}")

## 2 · Lanzador (reanudable, log completo a fichero + resumen en vivo) y estado persistente
* `run_cmd` guarda **todo** el log en `logs/*.log` y solo imprime un resumen cada `show_every` iteraciones.
* `train_one` da el entrenamiento por bueno solo si `rc==0` **y** no hay `Traceback` **y** existe el checkpoint
  (isaaclab.bat puede devolver rc=0 aunque train.py haya fallado).
* `save_state/load_state` evitan perder `stage2/abl/nodr` si se reinicia el kernel.

In [ ]:
registry_csv = path_resultados / "run_registry.csv"

def git_state():
    try:
        sh = lambda c: subprocess.run(c, cwd=ROOT, capture_output=True, text=True).stdout.strip()
        return {"git_commit": sh(["git", "rev-parse", "HEAD"]), "git_dirty": bool(sh(["git", "status", "--porcelain"]))}
    except Exception:
        return {"git_commit": "", "git_dirty": None}

OVERRIDE_VARS = ("AEROTAXI_TAU_MS", "AEROTAXI_WIND_MS", "AEROTAXI_GUST_STD", "AEROTAXI_TRAJ", "HORIZON_MODE")
_KEEP = ("Mean reward", "Mean episode length", "Mean action noise", "Episode_Termination", "Metrics/", "Early Stopping")

def run_cmd(args, log_path: Path, env_extra: dict | None = None, show_every: int = 10):
    env = os.environ.copy()
    for k in OVERRIDE_VARS:
        env.pop(k, None)                                    # nunca heredar overrides de una celda anterior
    env.update({k: str(v) for k, v in (env_extra or {}).items()})
    log_path.parent.mkdir(parents=True, exist_ok=True)
    t0, show = time.time(), False
    p = subprocess.Popen([str(a) for a in args], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, env=env,
                         text=True, encoding="utf-8", errors="ignore", bufsize=1)
    try:
        with open(log_path, "w", encoding="utf-8") as f:
            for line in p.stdout:
                f.write(line)                               # el log completo siempre se guarda
                m = re.search(r"Learning iteration (\d+)/(\d+)", line)
                if m:
                    show = int(m.group(1)) % show_every == 0
                    if show:
                        print(f"\n[it {m.group(1)}/{m.group(2)}  {time.time() - t0:.0f}s]")
                elif show and any(k in line for k in _KEEP):
                    print("   ", line.strip())
        rc = p.wait()
    except KeyboardInterrupt:
        p.kill(); raise                                     # parar la celda también para el entrenamiento
    return rc, time.time() - t0

def register(row: dict):
    row = {**row, "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"), **git_state()}
    pd.DataFrame([row]).to_csv(registry_csv, mode="a", header=not registry_csv.exists(), index=False)

def train_one(index: int, seed: int, env_extra: dict | None = None, tag: str = ""):
    ok = path_resultados / "done" / f"train_{index}{tag}.ok"
    if ok.exists():
        return "skipped"
    run = f"ppo_random_{experiment_name}_{index}"
    log = path_resultados / "logs" / f"train_{index}{tag}.log"
    args = [isaaclab_bat, "-p", train_script, "--task", f"{id_task}-{index}",
            "--num_envs", NUM_ENVS_TRAIN, "--seed", seed, "--headless"]
    rc, dt = run_cmd(args, log, env_extra)
    text = log.read_text(encoding="utf-8", errors="ignore")
    has_tb = "Traceback (most recent call last)" in text
    has_ckpt = any(list(d.glob(f"best_model_{run}_*.pt")) for d in (path_resultados, path_ral))
    good = (rc == 0) and not has_tb and has_ckpt
    register({"phase": "train", "index": index, "seed": seed, "tag": tag, "rc": rc, "traceback": has_tb,
              "checkpoint": has_ckpt, "seconds": round(dt), **{f"env_{k}": v for k, v in (env_extra or {}).items()}})
    if good:
        ok.parent.mkdir(exist_ok=True); ok.write_text("ok")
        return 0
    print(f"\n--- FALLO config {index} (rc={rc}, traceback={has_tb}, checkpoint={has_ckpt}) · últimas líneas de {log.name}:")
    print("\n".join(text.splitlines()[-25:]))
    return rc or 1

def save_state(**kw):
    (path_ral / "state.json").write_text(json.dumps(kw, indent=1))

def load_state():
    st = json.loads((path_ral / "state.json").read_text())
    tup = lambda d: {int(k): tuple(v) for k, v in d.items()}
    return st["shortlist"], st["best_cfg"], tup(st["stage2"]), tup(st["abl"]), tup(st.get("nodr", {}))

## 3 · Smoke test de entrenamiento (índice 100, carpeta `_smoke`; no toca el grid)

In [ ]:
smoke_dir = path_resultados / "_smoke"
write_cfg(SMOKE_INDEX, grid["1"]["hp"], seed=1, csv_path=smoke_dir, num_iter=3, patience=0)
log = path_resultados / "logs" / "smoke.log"
rc, _ = run_cmd([isaaclab_bat, "-p", train_script, "--task", f"{id_task}-{SMOKE_INDEX}", "--num_envs", 64,
                 "--max_iterations", 3, "--seed", 1, "--headless"], log, show_every=1)
txt = log.read_text(encoding="utf-8", errors="ignore")
print("rc =", rc, "| traceback:", "Traceback" in txt)
print("\n".join(txt.splitlines()[-25:]))

## 4 · Smoke test de `play.py` (valida eval_utils, columnas y overrides)

In [ ]:
ck = next(smoke_dir.glob("final_model_*_64.pt"))
log = path_resultados / "logs" / "smoke_play.log"
rc, _ = run_cmd([isaaclab_bat, "-p", play_script, "--task", f"{id_task}-{SMOKE_INDEX}", "--num_envs", 64, "--checkpoint", ck,
                 "--headless", "--seed", 7, "--tag", "smoke", "--record_states", "--states_csv_dir", smoke_dir / "traj"], log,
                env_extra={"AEROTAXI_TAU_MS": 50})
txt = log.read_text(encoding="utf-8", errors="ignore")
print("rc =", rc, "| traceback:", "Traceback" in txt)
print("\n".join(txt.splitlines()[-12:]))
row = pd.read_csv(smoke_dir / "rewards_play_PPO.csv").iloc[-1]
print(row[["tag", "n_episodes", "success", "aerotaxi_tau_ms"]])
traj = pd.read_csv(next((smoke_dir / "traj").glob("tracking_3D_*.csv")))
assert row.n_episodes == 64 and float(row.aerotaxi_tau_ms) == 50
assert traj.env_id.nunique() == 64 and {"episode", "terminated"} <= set(traj.columns)
print("SMOKE OK · episodios:", traj.env_id.nunique(), "· pasos máx.:", traj.step.max() + 1)
# shutil.rmtree(smoke_dir, ignore_errors=True)   # descomenta cuando hayas mirado el resultado

## 5 · Etapa 1 — random search (1 semilla, 35 configs)
Si una configuración falla, imprime las últimas líneas del log y sigue con la siguiente.

In [ ]:
for i in range(1, N_CONFIGS + 1):
    rc = train_one(i, seed=grid[str(i)]["seed"])
    print(f"config {i:>2}: {'OK' if rc in (0, 'skipped') else f'FALLO rc={rc}'}")

## 6 · Shortlist — SOLO métricas de entrenamiento (nada de test)

In [ ]:
df_best = pd.read_csv(path_resultados / "reward_best_models.csv")
df_best = df_best[df_best["num_envs_train"] == NUM_ENVS_TRAIN].copy()          # ignora restos de smoke
df_best["success_train"] = df_best["mean_Episode_Termination/time_out"]
df_best["cfg"] = df_best["id_run_name"].str.extract(r"_(\d+)$")[0].astype(int)
shortlist = (df_best[df_best["success_train"] > 0.85]
             .sort_values("reward", ascending=False).head(TOP_K)["cfg"].tolist())
assert shortlist, "ninguna config supera success_train>0.85: revisa el umbral"
print("Configs que pasan a etapa 2:", shortlist)
display(df_best.sort_values("reward", ascending=False)[["cfg", "reward", "std_reward", "success_train", "iteration"]].head(10))

## 7 · Etapa 2 — multi-semilla de las configs preseleccionadas

In [ ]:
stage2 = {}                                     # index -> (cfg_base, seed)
idx = 36
for base in shortlist:
    for s in TRAIN_SEEDS:
        write_cfg(idx, grid[str(base)]["hp"], seed=s, csv_path=path_ral, num_iter=NUM_ITER_STAGE2, patience=0)
        stage2[idx] = (base, s); idx += 1
assert idx - 1 < SMOKE_INDEX
for i, (base, s) in stage2.items():
    print(i, base, s, train_one(i, seed=s))
save_state(shortlist=shortlist, best_cfg=shortlist[0], stage2=stage2, abl={}, nodr={})

## 8 · Ablación del horizonte de anticipación (misma config, 5 semillas por brazo)
`HORIZON_MODE` se pasa por entorno al entrenar **y** al evaluar (la observación depende de él).

In [ ]:
best_cfg = shortlist[0]
abl = {}
for mode in ("fixed", "none"):
    for s in TRAIN_SEEDS:
        write_cfg(idx, grid[str(best_cfg)]["hp"], seed=s, csv_path=path_ral, num_iter=NUM_ITER_STAGE2, patience=0)
        abl[idx] = (best_cfg, s, mode); idx += 1
assert idx - 1 < SMOKE_INDEX
for i, (base, s, mode) in abl.items():
    print(i, mode, s, train_one(i, seed=s, env_extra={"HORIZON_MODE": mode}, tag=f"_{mode}"))
save_state(shortlist=shortlist, best_cfg=best_cfg, stage2=stage2, abl=abl, nodr={})

## 9 · Brazo sin aleatorización (no-DR): τ=0 y sin viento en entrenamiento
Sin esta referencia no se puede afirmar que la aleatorización aporte robustez.
La etiqueta `"nodr"` viaja por `HORIZON_MODE`; el entorno solo trata `fixed`/`none` de forma especial
(cualquier otro valor = horizonte estocástico).

In [ ]:
nodr = {}
for s in TRAIN_SEEDS:
    write_cfg(idx, grid[str(best_cfg)]["hp"], seed=s, csv_path=path_ral, num_iter=NUM_ITER_STAGE2, patience=0)
    nodr[idx] = (best_cfg, s, "nodr"); idx += 1
assert idx - 1 < SMOKE_INDEX
NODR_ENV = {"AEROTAXI_TAU_MS": 0, "AEROTAXI_WIND_MS": 0, "AEROTAXI_GUST_STD": 0, "HORIZON_MODE": "stochastic"}
for i, (b, s, m) in nodr.items():
    print(i, s, train_one(i, seed=s, env_extra=NODR_ENV, tag="_nodr"))
save_state(shortlist=shortlist, best_cfg=best_cfg, stage2=stage2, abl=abl, nodr=nodr)

## 10 · (Solo tras reiniciar el kernel) recargar el estado
Ejecuta antes las celdas 0, 1 y 2 y después esta.

In [ ]:
# shortlist, best_cfg, stage2, abl, nodr = load_state()
# print(len(stage2), len(abl), len(nodr))

## 11 · Evaluación — matriz de condiciones, semillas de test distintas, condiciones fuera de distribución
* `nominal` = τ=0 con el viento del entrenamiento (0–8 m/s + ráfagas).
* `tau*` y `wind12/16` superan el rango de entrenamiento (τ≤40 ms, viento≤8 m/s); `circle/lemniscate` nunca se ven al entrenar.
* Los brazos de ablación (`fixed`, `none`) se evalúan solo en `nominal`.

In [ ]:
CONDITIONS = (
    [dict(name="nominal", traj="random", tau_ms=0, wind_ms=None)]
    + [dict(name=f"tau{t}", traj="random", tau_ms=t, wind_ms=None) for t in (25, 50, 75, 100)]
    + [dict(name=f"wind{w}", traj="random", tau_ms=0, wind_ms=w) for w in (0, 12, 16)]
    + [dict(name=t, traj=t, tau_ms=0, wind_ms=None) for t in ("circle", "lemniscate")]
)

def eval_one(index, seed_train, horizon, cond, ckpt: Path):
    tag = f"idx{index}_s{seed_train}_{horizon}_{cond['name']}"
    out = path_ral / "eval" / tag
    if list(out.glob("tracking_3D_*.csv")):
        return "skipped"
    env_extra = {"AEROTAXI_TRAJ": cond["traj"], "HORIZON_MODE": horizon}
    if cond["tau_ms"] is not None: env_extra["AEROTAXI_TAU_MS"] = cond["tau_ms"]
    if cond["wind_ms"] is not None:
        env_extra["AEROTAXI_WIND_MS"] = cond["wind_ms"]
        env_extra["AEROTAXI_GUST_STD"] = 0 if cond["wind_ms"] == 0 else 2.0
    args = [isaaclab_bat, "-p", play_script, "--task", f"{id_task}-{index}", "--num_envs", N_TEST_ENVS,
            "--checkpoint", ckpt, "--headless", "--seed", EVAL_SEED, "--tag", tag,
            "--record_states", "--states_csv_dir", out]
    log = path_ral / "logs" / f"play_{tag}.log"
    rc, dt = run_cmd(args, log, env_extra)
    text = log.read_text(encoding="utf-8", errors="ignore")
    ok = (rc == 0) and "Traceback (most recent call last)" not in text and bool(list(out.glob("tracking_3D_*.csv")))
    register({"phase": "eval", "index": index, "seed_train": seed_train, "horizon": horizon, "cond": cond["name"],
              "seed_eval": EVAL_SEED, "rc": rc, "ok": ok, "seconds": round(dt), **{f"env_{k}": v for k, v in env_extra.items()}})
    if not ok:
        print(f"--- FALLO eval {tag}:\n" + "\n".join(text.splitlines()[-20:]))
    return 0 if ok else (rc or 1)

def ckpt_of(index):    # modelo de la ÚLTIMA iteración (sin selección por reward ruidoso)
    return next(path_ral.glob(f"final_model_ppo_random_{experiment_name}_{index}_*.pt"))

policies = ([(i, s, "stochastic", b) for i, (b, s) in stage2.items()]
            + [(i, s, m, b) for i, (b, s, m) in abl.items()]
            + [(i, s, "nodr", b) for i, (b, s, m) in nodr.items()])
for (i, s, h, b) in policies:
    conds = CONDITIONS if h in ("stochastic", "nodr") else CONDITIONS[:1]
    for c in conds:
        print(i, h, c["name"], eval_one(i, s, h, c, ckpt_of(i)))

## 12 · Métricas físicas por episodio (independientes de la recompensa)
* `success` = time-out sin terminación por fallo · `rmse_all` = RMSE hasta el fin o hasta el fallo
* `rmse_ss` = RMSE tras el transitorio de `BURN_IN_S`
* Comparar condiciones SIEMPRE con (success, rmse_ss, time_to_failure) a la vez: el RMSE solo sobre supervivientes
  mejora artificialmente si los peores episodios fallan.

In [ ]:
def episode_metrics(df: pd.DataFrame, dt=DT, burn_in_s=BURN_IN_S) -> pd.DataFrame:
    need = {"env_id", "episode", "step", "drone_x", "drone_y", "drone_z", "target_x", "target_y", "target_z", "terminated"}
    missing = need - set(df.columns)
    if missing:
        raise ValueError(f"Faltan columnas en el CSV de trayectorias: {sorted(missing)}")
    d = df.sort_values(["env_id", "episode", "step"]).copy()
    d["err"] = np.sqrt((d.drone_x - d.target_x) ** 2 + (d.drone_y - d.target_y) ** 2 + (d.drone_z - d.target_z) ** 2)
    d["t"] = d.groupby(["env_id", "episode"]).cumcount() * dt
    g = d.groupby(["env_id", "episode"])
    ep = g.agg(T=("t", "max"), failed=("terminated", "max"), rmse_all=("err", lambda e: float(np.sqrt(np.mean(e ** 2)))),
               emax=("err", "max")).reset_index()
    ss = d[d.t >= burn_in_s].groupby(["env_id", "episode"])["err"].apply(lambda e: float(np.sqrt(np.mean(e ** 2)))).rename("rmse_ss")
    ep = ep.merge(ss.reset_index(), on=["env_id", "episode"], how="left")
    ep["failed"] = ep["failed"].astype(bool)
    ep["success"] = ~ep["failed"]
    ep["time_to_failure"] = np.where(ep.failed, ep["T"], np.nan)
    return ep

def wilson(k, n, z=1.96):
    if n == 0: return (np.nan, np.nan)
    p = k / n; den = 1 + z ** 2 / n
    c = (p + z ** 2 / (2 * n)) / den; h = z * np.sqrt(p * (1 - p) / n + z ** 2 / (4 * n ** 2)) / den
    return (c - h, c + h)

def boot_ci(x, stat=np.mean, n=10000, alpha=0.05, seed=0):
    x = np.asarray(x, float); x = x[~np.isnan(x)]
    if len(x) < 2: return (np.nan, np.nan)
    rng = np.random.default_rng(seed)
    s = [stat(rng.choice(x, len(x), replace=True)) for _ in range(n)]
    return tuple(np.quantile(s, [alpha / 2, 1 - alpha / 2]))

def iqm(x):
    x = np.sort(np.asarray(x, float)); n = len(x)
    return float(x[int(np.floor(.25 * n)): int(np.ceil(.75 * n))].mean()) if n >= 4 else float(np.mean(x))

def load_all_episodes() -> pd.DataFrame:
    rows = []
    for csv in (path_ral / "eval").glob("*/tracking_3D_*.csv"):
        m = re.match(r"idx(\d+)_s(\d+)_(\w+?)_(\w+)$", csv.parent.name)
        index, seed_train, horizon, cond = int(m[1]), int(m[2]), m[3], m[4]
        ep = episode_metrics(pd.read_csv(csv))
        ep["index"], ep["seed_train"], ep["horizon"], ep["cond"] = index, seed_train, horizon, cond
        ep["cfg"] = (stage2.get(index) or abl.get(index) or nodr.get(index))[0]
        rows.append(ep)
    return pd.concat(rows, ignore_index=True)

## 13 · Agregación — la unidad estadística es la **semilla de entrenamiento**, no el episodio

In [ ]:
def per_policy(ep: pd.DataFrame) -> pd.DataFrame:
    return (ep.groupby(["cfg", "horizon", "cond", "seed_train"])
              .agg(success=("success", "mean"), rmse_ss=("rmse_ss", "median"), rmse_all=("rmse_all", "median"),
                   ttf=("time_to_failure", "median"), n_ep=("success", "size")).reset_index())

def summarize(pp: pd.DataFrame, by=("cfg", "horizon", "cond")) -> pd.DataFrame:
    out = []
    for key, g in pp.groupby(list(by)):
        lo_s, hi_s = boot_ci(g.success); lo_r, hi_r = boot_ci(g.rmse_ss)
        out.append(dict(zip(by, key)) | dict(n_seeds=len(g), success_pct=100 * g.success.mean(),
                        success_ci=f"[{100*lo_s:.1f}, {100*hi_s:.1f}]", rmse_ss_m=g.rmse_ss.mean(),
                        rmse_ss_ci=f"[{lo_r:.2f}, {hi_r:.2f}]", iqm_rmse_ss=iqm(g.rmse_ss)))
    return pd.DataFrame(out)

def paired_delta(pp, cond_a, cond_b, metric="rmse_ss", horizon="stochastic"):
    """Diferencia pareada por semilla (mismas políticas, distinta condición) + IC bootstrap."""
    q = pp[pp.horizon == horizon]
    a = q[q.cond == cond_a].set_index(["cfg", "seed_train"])[metric]
    b = q[q.cond == cond_b].set_index(["cfg", "seed_train"])[metric]
    d = (b - a).dropna()
    return d.mean(), boot_ci(d)

episodes = load_all_episodes()
pp = per_policy(episodes)
tab = summarize(pp)
display(tab)
for h in ("stochastic", "nodr"):
    if (pp.horizon == h).any():
        print(h, "· tau50 - nominal, RMSE_ss:", paired_delta(pp, "nominal", "tau50", horizon=h))
        print(h, "· tau50 - nominal, success :", paired_delta(pp, "nominal", "tau50", "success", horizon=h))

## 14 · Figuras con estilo IEEE (columna doble 7.16 in, PDF vectorial, fuentes ≥7 pt, paleta apta para daltónicos)

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
OKABE = ["#0072B2", "#D55E00", "#009E73", "#CC79A7", "#E69F00", "#56B4E9", "#000000"]
def ieee_style():
    mpl.rcParams.update({"font.size": 8, "font.family": "serif", "axes.labelsize": 8, "legend.fontsize": 7,
                         "xtick.labelsize": 7, "ytick.labelsize": 7, "axes.prop_cycle": mpl.cycler(color=OKABE),
                         "pdf.fonttype": 42, "ps.fonttype": 42, "axes.grid": True, "grid.alpha": .3, "grid.linestyle": ":",
                         "savefig.bbox": "tight", "savefig.dpi": 300})
ieee_style()

def curves(df_log: pd.DataFrame, run_names) -> pd.DataFrame:
    """Recorta a la iteración común mínima (evita el artefacto de media sobre n variable de semillas)."""
    cut = min(df_log[df_log.id_run_name == r].iteration.max() for r in run_names)
    return pd.concat([df_log[(df_log.id_run_name == r) & (df_log.iteration <= cut)]
                      .drop_duplicates("iteration").set_index("iteration").reward.rename(r) for r in run_names], axis=1)

def plot_band(ax, wide, label, color, ls="-"):
    wide = wide.rolling(5, min_periods=1).mean()                     # suaviza el ruido del buffer de episodios
    m = wide.mean(axis=1)
    lo, hi = (wide.quantile(0.025, axis=1), wide.quantile(0.975, axis=1)) if wide.shape[1] > 5 else (wide.min(axis=1), wide.max(axis=1))
    ax.plot(m.index, m, color=color, ls=ls, lw=1.2, label=f"{label} (n={wide.shape[1]})")
    ax.fill_between(m.index, lo, hi, color=color, alpha=.2, lw=0)

def err_bars(g, col, scale=1.0):
    mu = g[col].mean()
    lo = g[col].apply(lambda x: boot_ci(x)[0]); hi = g[col].apply(lambda x: boot_ci(x)[1])
    return mu * scale, [(mu - lo) * scale, (hi - mu) * scale]

log_ral = pd.concat([pd.read_csv(p) for p in (path_ral / "curves").glob("*.csv")], ignore_index=True)   # resolución: 10 it.
rn = lambda i: f"ppo_random_{experiment_name}_{i}"
arms = {"Stochastic [0.4, 0.9] s": [rn(i) for i, (b, s) in stage2.items() if b == best_cfg],
        "Fixed 0.65 s": [rn(i) for i, (b, s, m) in abl.items() if m == "fixed"],
        "No look-ahead": [rn(i) for i, (b, s, m) in abl.items() if m == "none"],
        "No DR": [rn(i) for i in nodr]}
fig, ax = plt.subplots(1, 3, figsize=(7.16, 2.3))
for (name, runs), c, ls in zip(arms.items(), OKABE, ("-", "--", "-.", ":")):
    if runs: plot_band(ax[0], curves(log_ral, runs), name, c, ls)
ax[0].set_xlabel("PPO iteration"); ax[0].set_ylabel("Training episode return"); ax[0].legend(loc="lower right", fontsize=6)

taus = {"nominal": 0, "tau25": 25, "tau50": 50, "tau75": 75, "tau100": 100}
for h, c, lab in (("stochastic", OKABE[0], "DR (proposed)"), ("nodr", OKABE[1], "No DR")):
    sub = pp[(pp.horizon == h) & pp.cond.isin(taus)].assign(tau=lambda d: d.cond.map(taus))
    if sub.empty: continue
    g = sub.groupby("tau")
    mu, yerr = err_bars(g, "rmse_ss");   ax[1].errorbar(mu.index, mu, yerr=yerr, marker="o", ms=3, capsize=2, color=c, label=lab)
    mu, yerr = err_bars(g, "success", 100); ax[2].errorbar(mu.index, mu, yerr=yerr, marker="s", ms=3, capsize=2, color=c, label=lab)
for a_, yl in ((ax[1], "Steady-state RMSE (m)"), (ax[2], "Success rate (%)")):
    a_.set_xlabel("Actuator time constant $\\tau$ (ms)"); a_.set_ylabel(yl); a_.axvspan(40, 100, color="grey", alpha=.08, lw=0)
ax[1].legend(loc="upper left", fontsize=6)
fig.tight_layout(); fig.savefig(path_ral / "fig_convergence_ablation_latency.pdf"); plt.show()

## 15 · Trayectorias cualitativas (episodio de RMSE **mediano**, no el mejor)

In [ ]:
def traj_fig(index, seed_train, horizon, conds=("nominal", "circle", "lemniscate"), name="fig_trajectories.pdf"):
    fig, axs = plt.subplots(1, len(conds), figsize=(7.16, 2.3))
    axs = np.atleast_1d(axs)
    for ax_, c in zip(axs, conds):
        d = pd.read_csv(next((path_ral / "eval" / f"idx{index}_s{seed_train}_{horizon}_{c}").glob("tracking_3D_*.csv")))
        ep = episode_metrics(d).dropna(subset=["rmse_ss"])
        env_id = ep.loc[(ep.rmse_ss - ep.rmse_ss.median()).abs().idxmin(), "env_id"]
        g = d[d.env_id == env_id]
        ax_.plot(g.target_x, g.target_y, "k--", lw=1, label="Reference")
        ax_.plot(g.drone_x, g.drone_y, lw=1.2, color=OKABE[0], label="PPO")
        ax_.set_title(c, fontsize=8); ax_.set_aspect("equal", adjustable="datalim"); ax_.set_xlabel("x (m)")
    axs[0].set_ylabel("y (m)"); axs[0].legend(loc="best")
    fig.tight_layout(); fig.savefig(path_ral / name); plt.show()

i0, (b0, s0) = next(iter(stage2.items()))
traj_fig(i0, s0, "stochastic")

## 16 · Tablas LaTeX (`booktabs`; copiar a `\input{}` en el paper)

In [ ]:
def make_tables(pp, out_dir=path_ral):
    tab = summarize(pp)
    conds = ["nominal", "tau25", "tau50", "tau75", "tau100", "wind0", "wind12", "wind16", "circle", "lemniscate"]
    t = tab[tab.cond.isin(conds) & tab.horizon.isin(["stochastic", "nodr"])].copy()
    t["Success (\\%)"] = t.success_pct.map("{:.1f}".format) + " " + t.success_ci
    t["RMSE$_{ss}$ (m)"] = t.rmse_ss_m.map("{:.2f}".format) + " " + t.rmse_ss_ci
    piv = t.pivot_table(index="cond", columns="horizon", values=["Success (\\%)", "RMSE$_{ss}$ (m)"], aggfunc="first")
    piv = piv.reindex([c for c in conds if c in piv.index])
    (Path(out_dir) / "table_robustness.tex").write_text(
        piv.to_latex(escape=False, caption="Success rate and steady-state RMSE (mean, 95\\% bootstrap CI over training seeds).",
                     label="tab:robustness"), encoding="utf-8")
    a = tab[tab.cond == "nominal"].copy()
    (Path(out_dir) / "table_horizon_ablation.tex").write_text(
        a[["horizon", "n_seeds", "success_pct", "success_ci", "rmse_ss_m", "rmse_ss_ci"]].to_latex(
            index=False, float_format="%.2f", caption="Look-ahead horizon ablation (nominal conditions).", label="tab:horizon"),
        encoding="utf-8")
    return piv

display(make_tables(pp))

## 17 · Manifiesto de reproducibilidad (material suplementario)

In [ ]:
def pip_freeze_subset():
    try:
        out = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True).stdout.splitlines()
        return [l for l in out if re.match(r"(torch|numpy|pandas|isaaclab|rsl|gymnasium|scipy)", l, re.I)]
    except Exception:
        return []

def gpu_info():
    try:
        return subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version", "--format=csv,noheader"],
                              capture_output=True, text=True).stdout.strip()
    except Exception:
        return ""

manifest = {"created": time.strftime("%Y-%m-%d %H:%M:%S"), **git_state(), "task": id_task, "env": "env_19",
            "num_envs_train": NUM_ENVS_TRAIN, "num_steps_per_env": NUM_STEPS_PER_ENV, "max_iterations_stage1": NUM_ITER,
            "max_iterations_stage2": NUM_ITER_STAGE2, "train_seeds": TRAIN_SEEDS, "eval_seed": EVAL_SEED,
            "n_test_envs": N_TEST_ENVS, "grid_seed": GRID_SEED, "shortlist": shortlist, "best_cfg": best_cfg,
            "conditions": CONDITIONS, "packages": pip_freeze_subset(), "gpu": gpu_info()}
(path_ral / "manifest.json").write_text(json.dumps(manifest, indent=1, default=str))
print(json.dumps({k: manifest[k] for k in ("git_commit", "git_dirty", "gpu")}, indent=1))